[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/22_conv2d.ipynb)

# 🟠 Medium: 2D Convolution

Implement **2D convolution** from scratch.

### Signature
```python
def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    # x: (B, C_in, H, W), weight: (C_out, C_in, kH, kW)
    # Returns: (B, C_out, H_out, W_out)
```

### Rules
- Do NOT use `F.conv2d` or `nn.Conv2d`
- Support `stride` and `padding` parameters
- `F.pad` for zero-padding is allowed

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.8 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn.functional as F

In [34]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    # extract patches, apply kernel, handle stride/padding
    bs, _, h_in, w_in = x.size()
    c_out, _, h_k, w_k = weight.size()
    h_out = 1 + (h_in + 2 * padding - h_k) // stride
    w_out = 1 + (w_in + 2 * padding - w_k) // stride

    z = torch.zeros(bs, c_out, h_out, w_out)
    x_padded = F.pad(x, (padding, padding, padding, padding), mode='constant', value=0)
    for i in range(h_out): # traverses output matrix
        for j in range(w_out):
            i_in = i * stride
            j_in = j * stride
            window = x_padded[:, :, i_in:i_in+h_k, j_in:j_in+w_k]
            x_expand = window.unsqueeze(1) # b, 1, c, h, w
            k_expand = weight.unsqueeze(0) # 1, o, i, h, w
            xk = x_expand * k_expand
            z[:, :, i, j] = torch.sum(xk, dim=[2, 3, 4])
    
    if bias is not None:
        z += bias.view(1, -1, 1, 1)
    return z

In [35]:
# 🧪 Debug
x = torch.randn(1, 3, 8, 8)
w = torch.randn(16, 3, 3, 3)
print('Output:', my_conv2d(x, w).shape)
print('Match:', torch.allclose(my_conv2d(x, w), F.conv2d(x, w), atol=1e-4))

Output: torch.Size([1, 16, 6, 6])
Match: True


In [36]:
# ✅ SUBMIT
from torch_judge import check
check('conv2d')


🧪 Testing: 2D Convolution (Medium)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (4.5ms)
  ✅ [2/5] Matches F.conv2d (110.0ms)
  ✅ [3/5] With padding (2.2ms)
  ✅ [4/5] With stride (1.5ms)
  ✅ [5/5] Gradient flow (1.0ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (119.3ms total)
  Progress saved. Run status() to see your dashboard.

